<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/transcription/10_Note_Debouncing_and_Grouping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================
# [Bass Separator] Ultimate Environment Setup (v4.0 - Best Practice)
# ==================================================================
import os
import sys
import subprocess
from google.colab import drive

print("🚀 Bass Separator 통합 환경 설정을 시작합니다...")

# -----------------------------------------------------------------
# 1. Google Drive 마운트
# -----------------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# -----------------------------------------------------------------
# 2. GitHub 최신화 및 경로 설정 (충돌 없는 강제 동기화)
# -----------------------------------------------------------------
PROJECT_NAME = "Bass-separator"
REPO_URL = "https://github.com/sjkim-audio/Bass-separator.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

try:
    if not os.path.exists(PROJECT_PATH):
        print(f"📦 레포지토리 클론 중... ({PROJECT_NAME})")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    else:
        print(f"🔄 레포지토리 최신화 중... (Git Fetch & Reset)")
        subprocess.run(["git", "fetch", "--all"], cwd=PROJECT_PATH, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PROJECT_PATH, check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"❌ Git 동기화 실패: {e}")

# 핵심: 작업 디렉토리를 프로젝트 루트로 완벽히 고정하여 requirements.txt를 찾게 함
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

# -----------------------------------------------------------------
# 3. 커스텀 모듈(src) 실행을 통한 의존성 설치 및 데이터셋 로드
# -----------------------------------------------------------------
try:
    from src.env_setup import init_colab_env
    from src.utils import load_data_from_drive

    # env_setup.py 내부의 함수를 호출하여 requirements.txt 기반 설치 실행
    # (이 단계에서 torchcrepe, pretty_midi 등이 정상 설치됩니다)
    init_colab_env()

    # 데이터셋 복사
    MY_DRIVE_DATA_PATH = "/content/drive/MyDrive/Bass_separator/dataset"
    load_data_from_drive(MY_DRIVE_DATA_PATH, force_update=False)

except ImportError as e:
    print(f"⚠️ 커스텀 모듈 임포트 에러: {e}")
    print("   (src/utils.py의 'import shutil' 오타가 수정되었는지 확인하세요!)")
except Exception as e:
    print(f"❌ 셋업 중단: {e}")

# -----------------------------------------------------------------
# 4. 글로벌 라이브러리 사전 적재
# -----------------------------------------------------------------
import librosa
import numpy as np
import pandas as pd
import torchcrepe
import pretty_midi

print("\n🎉 Ready to Rock! 모든 셋업과 모듈 로드가 완벽히 끝났습니다.")

🚀 Bass Separator 통합 환경 설정을 시작합니다...
Mounted at /content/drive
📦 레포지토리 클론 중... (Bass-separator)
🚀 환경 설정을 시작합니다...

🔧 [시스템] 필수 도구 확인 중...
✅ FFmpeg가 이미 설치되어 있습니다.

🐍 [파이썬] 라이브러리 확인 중...
📄 requirements.txt 파일을 발견했습니다. 의존성 패키지를 설치합니다...
📦 패키지 일괄 설치 진행 중...
✅ 패키지 일괄 설치 완료.

🏥 설치 무결성 점검 (Health Check)...
✅ 필수 라이브러리가 모두 정상적으로 준비되었습니다!

🎉 모든 환경 설정이 완료되었습니다!
🚀 데이터 복사 시작...
   📂 Source: /content/drive/MyDrive/Bass_separator/dataset
   📂 Dest  : ./dataset
🎉 데이터 준비 완료! (총 5개 파일 복사됨)

🎉 Ready to Rock! 모든 셋업과 모듈 로드가 완벽히 끝났습니다.


In [2]:
import os
import librosa
from src.bass_transcription import get_f0_crepe_robust
from src.tab_generator import BassTabGenerator

# 1. 파일 경로 설정 및 오디오 로드
audio_path = '/content/drive/MyDrive/Bass_separator/dataset/performance_test_demo(bass).wav'

if not os.path.exists(audio_path):
    raise FileNotFoundError(f"❌ 파일을 찾을 수 없습니다: {audio_path}")

print(f"📂 오디오 로드 중: {os.path.basename(audio_path)}")
y, sr = librosa.load(audio_path, sr=16000)

# 2. 피치 트래킹 (CREPE Tiny 모델)
print("🚀 피치 트래킹 실행 중...")
f0_data = get_f0_crepe_robust(y, sr, hop_length=160, model_capacity='tiny', batch_size=512)

# 3. 타브 악보 생성기 초기화 및 디바운싱 파싱 (핵심 변경점)
print("📝 디바운싱(Debouncing) 적용 노트 파싱 중...")
tab_gen = BassTabGenerator(sr=16000, hop_length=160)
# min_duration_frames=5 (0.05초 이하 노이즈 무시), tolerance_frames=3 (찰나의 끊김 무시)
tab_gen.parse_f0_to_events(f0_data, min_duration_frames=5, tolerance_frames=3)

# 4. Viterbi 디코더 실행
print("🧠 Viterbi 기반 스마트 운지법 최적화 중...")
tab_gen.optimize_fingering()

# 5. 결과 렌더링
print("✨ 최적화 완료! 타브 악보를 렌더링합니다.")
tab_gen.display_tab(chars_per_line=80)

📂 오디오 로드 중: performance_test_demo(bass).wav
🚀 피치 트래킹 실행 중...
⚠️ 경고: GPU가 감지되지 않아 연산이 매우 느려질 수 있습니다.
📝 디바운싱(Debouncing) 적용 노트 파싱 중...
🧠 Viterbi 기반 스마트 운지법 최적화 중...
✅ Viterbi 운지법 최적화 완료.
✨ 최적화 완료! 타브 악보를 렌더링합니다.

🎸 Generated Bass Tab (Standard Tuning G-D-A-E)

G |---------------------------------------------------------------------|
D |--0---------0--------------------------------------------------------|
A |-------------------------------------------3-----------------------0-|
E |--------------------3---------3-------------------------5-------4----|

G |---------------------------------------------------------------------|
D |-----0------------0--------------------------------------------------|
A |--0----------------------------------------------3---------3-------0-|
E |-----------------------3-------4--3--3-------------------------------|

G |------------------------------------------------------------------------------|
D |---------------0---------------0-----------0--3----3--0------